# Rung 12c — composite input, on the pod

**One variable against the matched control: the model sees a second, edge-enhanced view of the frame.**

This notebook is a thin launcher over `_models/pipeline_12c.py`. It does not itself
train anything — it builds a config and calls `run`, which drives the whole session as
one resumable, file-observable state machine. Everything a watcher (agy) needs is on
disk: `STATE.json`, `events.jsonl`, and a `DONE`/`FAILED`/`ABORTED` marker.

**Read `context/12-image-processing/CONTEXT.md` §12c PRE-REGISTRATION first.** The arms,
thresholds and stopping rules are fixed there before any number exists.

## Order of operations on the pod
1. **Dry run** (seconds, no GPU) — proves orchestration, gates, paths.
2. **Real run** — control arm first and alone; `gate_harness` decides if the composite arm runs.
3. agy watches `STATE.json`; on `finished && safe_to_shutdown` it powers the pod off.

In [ ]:
# One cell: locate the repo and import. No hand-typed paths.
import sys
from pathlib import Path

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not (REPO / 'src').is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(EXP_DIR / '_models'))

import pipeline_12c as P
from frame import runstate as rs
print('repo:', REPO)

## 1 — Dry run
Exercises the full state machine with `dry_run=True`: every stage returns stub evidence,
no GPU, no `swift`, no video decode. If this does not end `[DONE] 11/11`, fix it before
warming the GPU.

In [ ]:
import shutil
DRY = Path('/workspace/orena_12c_dry')
shutil.rmtree(DRY, ignore_errors=True)
res = P.run(P.PipelineConfig(root=DRY, dry_run=True))
print(res['status'], '->', rs.summary(DRY))
assert res['status'] == 'done', res

## 2 — The real run

🔴 **Before this cell:** confirm the split manifest path and that
`/workspace/frames_cache` is populated. `extra['split_manifest']` must point at the
frozen OOD split (`experiments/splits/frame_ood_v1.csv`), verified by its sha256.

The TODO-bodied stages (`export_*`, `train_*`, `eval_*`, `build_maps`, `report`) are
wired to the rung-02/06 modules **here on the pod** and smoked once with a tiny frac
before the real frac. Do not run the real frac until each stage's smoke is green — that
is the discipline `pipeline_12c` cannot enforce for you.

In [ ]:
cfg = P.PipelineConfig(
    root=Path('/workspace/orena_12c'),
    repo=REPO,
    frac=0.25,
    seed=20260721,
    # pre-registered — do NOT edit after seeing a number
    floor_margin_min=0.0,
    tier1_max_deficit=0.03,
    shutdown_on_failure=False,  # a crash keeps the GPU alive for the post-mortem
    extra={'split_manifest': REPO / 'experiments/splits/frame_ood_v1.csv'},
)
cfg

In [ ]:
# Launch. Resumable: re-run this cell after any interruption and it skips stamped stages.
# Watch from ANOTHER shell — this cell does not need to stay attached:
#     watch -n5 python -m frame.runstate /workspace/orena_12c
result = P.run(cfg)
print(result)
print(rs.summary(cfg.root))

## 3 — What agy reads

The watcher never parses scrollback. It polls one file and acts on two booleans.

```python
from frame import runstate as rs
s = rs.read('/workspace/orena_12c')
if s['finished'] and s['safe_to_shutdown']:
    shutdown_pod()          # success or a pre-registered abort
elif s['finished']:
    alert('crashed — GPU kept alive for the post-mortem')  # safe_to_shutdown False
elif rs.is_stale('/workspace/orena_12c'):
    alert('no heartbeat > 5 min — investigate')
```

`python -m frame.runstate /workspace/orena_12c` prints the same state as one line.